# HR Quality Check Summary Generator

This notebook processes HR quality check CSV files in a specified folder and generates a summary table formatted with `Total Count ± Standard Deviation` for various HR ranges.

**Note:** Now includes a directory picker for modularity and the source filename in the top row.

In [ ]:
import pandas as pd
import os
import glob
import tkinter as tk
from tkinter import filedialog

def generate_hr_summary(folder_path):
    # List of features to extract
    features = [
        'HR Range: 0–40 bpm', 'HR Range: 40–60 bpm', 'HR Range: 60–80 bpm', 
        'HR Range: 80–100 bpm', 'HR Range: 100–120 bpm', 'HR Range: 120–140 bpm', 
        'HR Range: 140–160 bpm', 'HR Range: 160–180 bpm', 'HR Range: 180–200 bpm', 
        'HR Range: >200 bpm'
    ]

    # Mapping file substrings to the target columns
    mapping = {
        'CareWear_heart_rate': 'CareWear Galaxy Watch',
        'CareWear_biopac': 'CareWear Biopac',
        'CareWear_belt': 'CareWear Belt',
        'GalaxyPPG_heart_rate': 'GalaxyPPG Galaxy Watch',
        'GalaxyPPG_E4': 'GalaxyPPG E4',
        'GalaxyPPG_Polar': 'GalaxyPPG Polar H10'
    }

    results = {}
    csv_files = glob.glob(os.path.join(folder_path, '*.csv'))
    
    for file_path in csv_files:
        filename = os.path.basename(file_path)
        if 'HR_Range_Summary' in filename or 'automated' in filename.lower(): 
            continue
            
        try:
            df = pd.read_csv(file_path)
            
            # Determine device mapping
            device_target = "Unknown"
            for pattern, name in mapping.items():
                if pattern in filename:
                    device_target = name
                    break
            
            if device_target == "Unknown":
                # Try to use filename if no mapping found
                device_target = filename.split('_heart_rate_quality')[0]
                
            file_stats = {}
            for feat in features:
                if feat in df.columns:
                    total_count = df[feat].sum()
                    std_val = df[feat].std()
                    file_stats[feat] = f"{int(total_count)} ± {std_val:.2f}"
                else:
                    file_stats[feat] = ""
            
            results[device_target] = {"filename": filename, "stats": file_stats}
            
        except Exception as e:
            print(f"Error processing {filename}: {e}")

    if not results:
        print("No relevant CSV files found in: ", folder_path)
        return

    # Standard device columns
    standard_cols = [
        'CareWear Galaxy Watch', 'CareWear Belt', 'CareWear Biopac',
        'GalaxyPPG Galaxy Watch', 'GalaxyPPG E4', 'GalaxyPPG Polar H10'
    ]
    
    # Add any extra devices found in the folder
    all_devices = sorted(list(set(standard_cols) | set(results.keys())))
    # Only keep devices that actually have results
    column_names = [c for c in all_devices if c in results]

    # Build filenames row and data rows
    filenames_row = {}
    data_rows = {feat: {} for feat in features}
    
    for col in column_names:
        filenames_row[col] = results[col]["filename"]
        for feat in features:
            data_rows[feat][col] = results[col]["stats"][feat]

    # Create main dataframe flipped to match target structure
    summary_df = pd.DataFrame(data_rows).T[column_names]
    
    # Prepend the filename row as a data row
    summary_df.loc['source_filename'] = pd.Series(filenames_row)
    
    # Reorder to have 'source_filename' at the top index
    summary_df = summary_df.reindex(['source_filename'] + features)
    
    # Set the index name to 'filename' as per requirement
    summary_df.index.name = 'filename'
    
    # Write to CSV
    output_path = os.path.join(folder_path, 'HR_Range_Summary_Automated.csv')
    summary_df.to_csv(output_path)
    print(f"Results written to: {output_path}")
    
    return summary_df

# Launch Tkinter Directory Picker
root = tk.Tk()
root.withdraw()
root.lift()
root.attributes('-topmost', True)
root.update()

try:
    input_folder = filedialog.askdirectory(title="Select Folder containing Quality Check CSVs")
except Exception as e:
    print(f"Dialog error: {e}")
    input_folder = None

root.destroy()

# Fallback to manual entry if dialog is cancelled or fails
if not input_folder:
    print("No folder selected via popup.")
    input_folder = input("Please manually enter the folder path (or leave blank to cancel): ").strip()

if input_folder and os.path.isdir(input_folder):
    summary = generate_hr_summary(input_folder)
    if 'display' in globals():
        display(summary)
    else:
        print(summary)
elif input_folder:
    print(f"Error: '{input_folder}' is not a valid directory.")
else:
    print("Operation cancelled.")

2026-03-24 13:07:21.186 python[16839:84414] The class 'NSOpenPanel' overrides the method identifier.  This method is implemented by class 'NSWindow'


Results written to: /Volumes/ss/Project_CareWear/DATASET/ss_drive/quality-check/HR_Range_Summary_Automated.csv
                                                         CareWear Biopac  \
filename                                                                   
source_filename        CareWear_biopac_heart_rate_quality_features_60...   
HR Range: 0–40 bpm                                        97818 ± 104.80   
HR Range: 40–60 bpm                                     975062 ± 1193.49   
HR Range: 60–80 bpm                                    6486407 ± 2589.69   
HR Range: 80–100 bpm                                   5455881 ± 2465.62   
HR Range: 100–120 bpm                                  1122709 ± 1344.78   
HR Range: 120–140 bpm                                    430115 ± 604.67   
HR Range: 140–160 bpm                                    353046 ± 353.18   
HR Range: 160–180 bpm                                    749913 ± 693.29   
HR Range: 180–200 bpm                                